# Code-Mixed Pedagogical Flow Extractor
## Subtask 2: PARALLEL Audio Extraction & Transcription (Dual T4 GPU)

**Instructions for Kaggle:**
1. Under the **Settings** panel on the right, ensure the **Accelerator** is set to `GPU T4 x2`.
2. Make sure your `irel-metadata` dataset containing `metadata.json` is uploaded.
3. Check the `METADATA_PATH` in the second cell.
4. Click **Run All**.
5. This notebook uses multiprocessing to load two separate Whisper models (one on `cuda:0`, one on `cuda:1`) to transcribe 2 videos simultaneously!

In [ ]:
!pip install -U yt-dlp openai-whisper ffmpeg-python

In [ ]:
import json
import os
import subprocess
import torch
import whisper
import concurrent.futures

# => IMPORTANT: Update this path to match where your dataset is uploaded! <=
METADATA_PATH = '/kaggle/input/datasets/akshatatkaggle/irel-task-metadata-subtask1/metadata.json'
OUTPUT_DIR = '/kaggle/working/data/interim'
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

num_gpus = torch.cuda.device_count()
print(f"Detected {num_gpus} GPUs on this Kaggle runner.")
if num_gpus < 2:
    print("WARNING: Less than 2 GPUs detected. This parallel script will run sequentially on the single available device.")

# Pre-load two separate Whisper models into VRAM (one on GPU 0, one on GPU 1)
print("Loading Whisper large-v3 onto cuda:0...")
model_0 = whisper.load_model('large-v3', device='cuda:0' if num_gpus > 0 else 'cpu')

print("Loading Whisper large-v3 onto cuda:1...")
model_1 = whisper.load_model('large-v3', device='cuda:1' if num_gpus > 1 else ('cuda:0' if num_gpus > 0 else 'cpu'))

def process_video(task_item):
    video_info, gpu_idx = task_item
    
    video_id = video_info['video_id']
    url = video_info['video_url']
    
    # Select the model pre-loaded onto the assigned GPU
    model = model_0 if gpu_idx == 0 else model_1
    device_name = f'cuda:{gpu_idx}' if num_gpus > 1 else ('cuda:0' if num_gpus > 0 else 'cpu')

    print(f'\n[GPU {gpu_idx}] >> Starting: {video_id}')
    
    audio_file = f'/kaggle/working/{video_id}.webm'
    wav_file = f'/kaggle/working/{video_id}.wav'
    
    # 1. Download audio 
    subprocess.run(['yt-dlp', '-f', 'bestaudio', '-o', audio_file, url], capture_output=True)
    if not os.path.exists(audio_file):
        audio_file = f'/kaggle/working/{video_id}.m4a'
        subprocess.run(['yt-dlp', '-f', 'm4a', '-o', audio_file, url], capture_output=True)
        
    # 2. Convert to 16kHz mono
    subprocess.run(['ffmpeg', '-y', '-i', audio_file, '-ar', '16000', '-ac', '1', wav_file], capture_output=True)
    
    if not os.path.exists(wav_file):
        return f"[GPU {gpu_idx}] Error: Failed to process audio for {video_id}"
        
    # 3. Transcribe!
    print(f'[GPU {gpu_idx}] Transcription running on {device_name} for {video_id}...')
    result = model.transcribe(wav_file, language='hi', task='transcribe', word_timestamps=True)
    
    # 4. Save
    out_path = os.path.join(OUTPUT_DIR, f'v1_raw_transcript_{video_id}.json')
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
        
    # Cleanup
    if os.path.exists(audio_file): os.remove(audio_file)
    if os.path.exists(wav_file): os.remove(wav_file)
    
    return f'[GPU {gpu_idx}] << Finished: {video_id}'

# Assign a GPU roughly via round-robin (0, 1, 0, 1...)
tasks = []
for i, vid in enumerate(metadata):
    gpu_assignment = i % 2 if num_gpus > 1 else 0
    tasks.append((vid, gpu_assignment))

print('\n========================================')
print(f'Starting Multiprocessing Pool with {min(num_gpus, 2) if num_gpus > 0 else 1} workers...')

# Start the ThreadPool
with concurrent.futures.ThreadPoolExecutor(max_workers=min(num_gpus, 2) if num_gpus > 0 else 1) as executor:
    # Submit all tasks
    futures = [executor.submit(process_video, task) for task in tasks]
    
    for future in concurrent.futures.as_completed(futures):
        try:
            msg = future.result()
            print(msg)
        except Exception as e:
            print(f"An error occurred during multi-processing: {e}")

print('\n========================================')
print('All Subtask 2 parallel transcriptions completed!')
print('Please download the 5 output JSON files from the /kaggle/working/data/interim folder.')
